In [1]:
import pandas as pd

In [2]:
import sqlite3

In [3]:
import os

In [4]:
df = pd.read_csv('../data/processed/rossmann_cleaned.csv', parse_dates=['Date'])

In [5]:
print(df.shape)

(844338, 18)


In [6]:
print(df.columns.tolist())

['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


In [7]:
## --- DIMENSION: dim_date ---
dim_date = df[['Date']].drop_duplicates().copy()
dim_date['date_id'] = range(1, len(dim_date) + 1)
dim_date['Year'] = dim_date['Date'].dt.year
dim_date['Month'] = dim_date['Date'].dt.month
dim_date['Week'] = dim_date['Date'].dt.isocalendar().week.astype(int)
dim_date['DayOfWeek'] = dim_date['Date'].dt.dayofweek
dim_date['IsWeekend'] = dim_date['DayOfWeek'].isin([5, 6]).astype(int)
dim_date['Quarter'] = dim_date['Date'].dt.quarter

In [8]:
## --- DIMENSION: dim_store ---
store_cols = ['Store','StoreType','Assortment','CompetitionDistance',
              'CompetitionOpenSinceMonth','CompetitionOpenSinceYear',
              'Promo2','Promo2SinceWeek','Promo2SinceYear','PromoInterval']
dim_store = df[store_cols].drop_duplicates(subset=['Store']).copy()

In [9]:
## --- DIMENSION: dim_promotion ---
dim_promo = df[['Promo','StateHoliday','SchoolHoliday']].drop_duplicates().copy()
dim_promo['promo_id'] = range(1, len(dim_promo) + 1)

In [10]:
## --- FACT TABLE: fact_sales ---
fact_sales = df[['Date','Store','Sales','Customers','Promo',
                 'StateHoliday','SchoolHoliday']].copy()

In [11]:
print("dim_date:", dim_date.shape)

dim_date: (942, 8)


In [12]:
print("dim_store:", dim_store.shape)

dim_store: (1115, 10)


In [13]:
print("dim_promo:", dim_promo.shape)

dim_promo: (13, 4)


In [14]:
print("fact_sales:", fact_sales.shape)

fact_sales: (844338, 7)


In [15]:
DB_PATH = '../data/processed/rossmann_warehouse.db'

In [16]:
conn = sqlite3.connect(DB_PATH)

In [17]:
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_store.to_sql('dim_store', conn, if_exists='replace', index=False)
dim_promo.to_sql('dim_promo', conn, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

844338

In [18]:
print("Tables saved to SQLite database.")

Tables saved to SQLite database.


In [19]:
## Verify by querying back
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM fact_sales", conn)
print(result)

   total_rows
0      844338


In [20]:
conn.close()

In [21]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [22]:
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_store ON fact_sales(Store)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_date ON fact_sales(Date)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_promo ON fact_sales(Promo)")

In [23]:
conn.commit()
conn.close()
print("Indexes created.")

Indexes created.


In [24]:
df = pd.read_csv('../data/processed/rossmann_cleaned.csv', parse_dates=['Date'])
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

In [25]:
## --- TIME FEATURES ---
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfMonth'] = df['Date'].dt.day
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
df['Quarter'] = df['Date'].dt.quarter

In [26]:
## --- LAG FEATURES (previous sales) ---
## Sort by store and date first!
df['Sales_Lag7'] = df.groupby('Store')['Sales'].shift(7)   # sales 7 days ago
df['Sales_Lag14'] = df.groupby('Store')['Sales'].shift(14)  # sales 14 days ago
df['Sales_Lag30'] = df.groupby('Store')['Sales'].shift(30)  # sales 30 days ago

In [27]:
## --- ROLLING AVERAGES ---
df['Sales_MA7'] = (df.groupby('Store')['Sales']
                   .transform(lambda x: x.shift(1).rolling(7).mean()))
df['Sales_MA30'] = (df.groupby('Store')['Sales']
                    .transform(lambda x: x.shift(1).rolling(30).mean()))

In [28]:
print("New columns added:", ['Year','Month','Week','DayOfMonth','IsWeekend',
      'Quarter','Sales_Lag7','Sales_Lag14','Sales_Lag30','Sales_MA7','Sales_MA30'])
print("NaN from lags (expected):", df['Sales_Lag7'].isna().sum())

New columns added: ['Year', 'Month', 'Week', 'DayOfMonth', 'IsWeekend', 'Quarter', 'Sales_Lag7', 'Sales_Lag14', 'Sales_Lag30', 'Sales_MA7', 'Sales_MA30']
NaN from lags (expected): 7805


In [29]:
## --- STORE-LEVEL AGGREGATES ---
store_avg = df.groupby('Store')['Sales'].mean().rename('Store_AvgSales')
store_std = df.groupby('Store')['Sales'].std().rename('Store_StdSales')
df = df.join(store_avg, on='Store')
df = df.join(store_std, on='Store')

In [30]:
## --- PROMO INTERACTION ---
df['Promo_Weekend'] = df['Promo'] * df['IsWeekend']

In [31]:
## --- ENCODE CATEGORICALS ---
df['StoreType_enc'] = df['StoreType'].map({'a':0,'b':1,'c':2,'d':3})
df['Assortment_enc'] = df['Assortment'].map({'a':0,'b':1,'c':2})
df['StateHoliday_enc'] = df['StateHoliday'].map({'none':0,'a':1,'b':2,'c':3})

In [32]:
## --- COMPETITION FEATURES ---
df['HasCompetition'] = (df['CompetitionDistance'] < 1000).astype(int)
df['CompetitionDistance_log'] = df['CompetitionDistance'].apply(
    lambda x: 0 if x == 0 else __import__('math').log1p(x))

In [33]:
print("Feature engineering complete!")
print("Total columns:", df.shape[1])
print(df[['Sales_Lag7','Sales_MA7','Store_AvgSales','StoreType_enc']].head())

Feature engineering complete!
Total columns: 37
   Sales_Lag7  Sales_MA7  Store_AvgSales  StoreType_enc
0         NaN        NaN     4759.096031              2
1         NaN        NaN     4759.096031              2
2         NaN        NaN     4759.096031              2
3         NaN        NaN     4759.096031              2
4         NaN        NaN     4759.096031              2


In [34]:
df.to_csv('../data/processed/rossmann_features.csv', index=False)

In [35]:
print("Saved rossmann_features.csv")

Saved rossmann_features.csv
